# Advanced Custom JSON Serialization — Tutorial-Style Problems with Solutions

This notebook develops **another set of advanced problems** on custom JSON serialization.

The teaching style is deliberately gradual:

- introduce one problem,
- inspect Python's default behavior,
- explain why it happens,
- make one small improvement,
- test it,
- then build toward a stronger solution.

The emphasis is on reasoning about serialization as a **data contract**, not just on making `json.dumps()` stop raising errors.

## What we will build

Across the problems we will explore:

- strict `default=` callbacks,
- typed/tagged JSON,
- `object_hook`,
- timezone-aware `datetime`,
- exact `Decimal` values,
- deterministic sets,
- UUIDs and bytes,
- `singledispatch`,
- nested custom objects,
- dataclasses,
- Enums,
- `JSONEncoder`,
- recursive decoding,
- versioned schemas,
- JSON Lines,
- safe class registries,
- type-tag collisions,
- tuple preservation,
- circular-reference limitations,
- and round-trip/failure tests.

In [1]:
import base64
import json
import math
import uuid

from dataclasses import dataclass, fields, is_dataclass
from datetime import datetime, timezone, timedelta
from decimal import Decimal
from enum import Enum
from functools import singledispatch
from io import StringIO
from pathlib import Path

# Problem 1 — Why `default=str` is often too lossy

Suppose a payload contains a `datetime` and a `Decimal`.

Python's default JSON encoder does not know how to serialize those types.

In [2]:
payload = {
    "created_at": datetime(2026, 8, 7, 10, 30, tzinfo=timezone.utc),
    "price": Decimal("10.50"),
}

Let us first confirm that the default encoder fails.

In [3]:
try:
    json.dumps(payload)
except TypeError as ex:
    print(type(ex).__name__, ex)

TypeError Object of type datetime is not JSON serializable


A common shortcut is:

```python
json.dumps(payload, default=str)
```

This makes the exception disappear, but it also removes the original type information.

In [4]:
text = json.dumps(payload, default=str, indent=2)
print(text)

{
  "created_at": "2026-08-07 10:30:00+00:00",
  "price": "10.50"
}


In [5]:
restored = json.loads(text)

print(restored)
print(type(restored["created_at"]))
print(type(restored["price"]))

{'created_at': '2026-08-07 10:30:00+00:00', 'price': '10.50'}
<class 'str'>
<class 'str'>


Both values are now ordinary strings.

So `default=str` is useful only when a string is truly the intended public representation. It is not a strong default for typed round trips.

# Problem 2 — Introduce explicit type tags

To reconstruct a value later, the JSON needs enough information to tell us what it used to be.

A common technique is to encode a special dictionary containing a type tag.

In [6]:
def encode_special(obj):
    if isinstance(obj, datetime):
        return {
            "__type__": "datetime",
            "value": obj.isoformat(),
        }

    if isinstance(obj, Decimal):
        return {
            "__type__": "decimal",
            "value": str(obj),
        }

    raise TypeError(f"Unsupported JSON type: {type(obj).__name__}")

Now serialize the same payload again.

In [7]:
typed_text = json.dumps(payload, default=encode_special, indent=2)
print(typed_text)

{
  "created_at": {
    "__type__": "datetime",
    "value": "2026-08-07T10:30:00+00:00"
  },
  "price": {
    "__type__": "decimal",
    "value": "10.50"
  }
}


The JSON is more verbose, but it now carries enough information for a decoder to distinguish a datetime from a decimal or a normal string.

# Problem 3 — Reconstruct tagged values with `object_hook`

`json.loads()` can call an `object_hook` for each dictionary it decodes.

That hook gives us a natural place to recognize the tags from the previous problem.

In [8]:
def decode_special(obj):
    tag = obj.get("__type__")

    if tag == "datetime":
        return datetime.fromisoformat(obj["value"])

    if tag == "decimal":
        return Decimal(obj["value"])

    return obj

In [9]:
restored = json.loads(typed_text, object_hook=decode_special)

print(restored)
print(type(restored["created_at"]))
print(type(restored["price"]))

{'created_at': datetime.datetime(2026, 8, 7, 10, 30, tzinfo=datetime.timezone.utc), 'price': Decimal('10.50')}
<class 'datetime.datetime'>
<class 'decimal.Decimal'>


Now verify that both value and type survive the round trip.

In [10]:
assert restored["created_at"] == payload["created_at"]
assert restored["price"] == payload["price"]
assert isinstance(restored["created_at"], datetime)
assert isinstance(restored["price"], Decimal)

print("Round trip passed.")

Round trip passed.


A useful property of `object_hook` is that nested dictionaries are processed before the dictionaries that contain them. We will use that later for nested custom objects.

# Problem 4 — Reject naive datetimes

The string `2026-08-07T10:30:00` does not say which timezone it belongs to.

For data interchange, that ambiguity is often unacceptable.

We will adopt a stricter rule:

> Every serialized datetime must be timezone-aware.

In [11]:
naive = datetime(2026, 8, 7, 10, 30)
aware = datetime(2026, 8, 7, 10, 30, tzinfo=timezone.utc)

print("naive:", naive.isoformat())
print("aware:", aware.isoformat())

naive: 2026-08-07T10:30:00
aware: 2026-08-07T10:30:00+00:00


Write a formatter that rejects naive values and normalizes everything else to UTC.

In [12]:
def datetime_to_utc_text(dt):
    if dt.tzinfo is None or dt.utcoffset() is None:
        raise ValueError("Naive datetime is not allowed.")

    utc_value = dt.astimezone(timezone.utc)
    return utc_value.isoformat().replace("+00:00", "Z")

In [13]:
print(datetime_to_utc_text(aware))

2026-08-07T10:30:00Z


In [14]:
try:
    datetime_to_utc_text(naive)
except ValueError as ex:
    print(type(ex).__name__, ex)

ValueError Naive datetime is not allowed.


Now test an explicit UTC+03:00 datetime.

In [15]:
plus_three = timezone(timedelta(hours=3))
local_time = datetime(2026, 8, 7, 15, 45, tzinfo=plus_three)

print(datetime_to_utc_text(local_time))

2026-08-07T12:45:00Z


15:45 at UTC+03:00 becomes 12:45 UTC.

Normalizing timestamps makes cross-system comparisons much easier.

# Problem 5 — Parse `Z` timestamps back to aware UTC datetimes

Our encoder now uses a `Z` suffix for UTC.

Python's `datetime.fromisoformat()` can easily parse an explicit offset, so we can translate `Z` to `+00:00` first.

In [16]:
def parse_utc_datetime(value):
    if value.endswith("Z"):
        value = value[:-1] + "+00:00"

    dt = datetime.fromisoformat(value)

    if dt.tzinfo is None or dt.utcoffset() is None:
        raise ValueError("Decoded datetime must be timezone-aware.")

    return dt.astimezone(timezone.utc)

In [17]:
encoded_time = datetime_to_utc_text(local_time)
decoded_time = parse_utc_datetime(encoded_time)

print("encoded:", encoded_time)
print("decoded:", decoded_time)

encoded: 2026-08-07T12:45:00Z
decoded: 2026-08-07 12:45:00+00:00


In [18]:
assert decoded_time == datetime(2026, 8, 7, 12, 45, tzinfo=timezone.utc)

# Problem 6 — Why converting `Decimal` to `float` can lose information

`Decimal` is often used specifically because binary floating-point values are not exact enough.

So converting a Decimal to float during serialization can undermine the original data model.

In [19]:
exact = Decimal("0.123456789012345678901234567890")
approx = float(exact)

print("Decimal:", exact)
print("float:  ", repr(approx))

Decimal: 0.123456789012345678901234567890
float:   0.12345678901234568


In [20]:
print(Decimal(str(approx)) == exact)

False


The comparison is false.

A better round-trip representation stores the decimal digits as text and uses a type tag to preserve the original type.

In [21]:
decimal_tag = {
    "__type__": "decimal",
    "value": str(exact),
}

decimal_tag

{'__type__': 'decimal', 'value': '0.123456789012345678901234567890'}

# Problem 7 — Replace a growing `if` chain with `singledispatch`

A custom formatter can quickly accumulate many `isinstance()` branches.

`functools.singledispatch` lets us register one function per type instead.

In [22]:
@singledispatch
def json_format(obj):
    raise TypeError(f"Unsupported JSON type: {type(obj).__name__}")

Register `datetime`.

In [23]:
@json_format.register(datetime)
def _(obj):
    return {
        "__type__": "datetime",
        "value": datetime_to_utc_text(obj),
    }

Register `Decimal`.

In [24]:
@json_format.register(Decimal)
def _(obj):
    return {
        "__type__": "decimal",
        "value": str(obj),
    }

Now the formatter is already modular: adding another type does not require editing one central `if` chain.

In [25]:
sample = {
    "time": datetime(2026, 8, 7, 12, 0, tzinfo=timezone.utc),
    "amount": Decimal("99.9900"),
}

print(json.dumps(sample, default=json_format, indent=2))

{
  "time": {
    "__type__": "datetime",
    "value": "2026-08-07T12:00:00Z"
  },
  "amount": {
    "__type__": "decimal",
    "value": "99.9900"
  }
}


# Problem 8 — Serialize sets, then make them deterministic

Sets are not JSON-native.

The obvious representation is a list, but there is a complication: sets are unordered.

If JSON text is used for hashing, caching, signatures, snapshots, or tests, arbitrary set iteration is undesirable.

In [26]:
@json_format.register(set)
def _(obj):
    return {
        "__type__": "set",
        "items": list(obj),
    }

In [27]:
roles = {"admin", "editor", "reviewer"}

print(json.dumps({"roles": roles}, default=json_format, indent=2))

{
  "roles": {
    "__type__": "set",
    "items": [
      "admin",
      "editor",
      "reviewer"
    ]
  }
}


For simple comparable elements we could call `sorted(obj)`.

But heterogeneous sets are more challenging because unrelated Python types may not be directly comparable.

A useful trick is to sort by each element's compact JSON representation.

In [28]:
def stable_sort_key(value):
    return json.dumps(
        value,
        default=json_format,
        sort_keys=True,
        separators=(",", ":"),
        ensure_ascii=False,
    )

In [29]:
@json_format.register(set)
def _(obj):
    return {
        "__type__": "set",
        "items": sorted(obj, key=stable_sort_key),
    }

# Problem 9 — Add UUID support and test a heterogeneous set

UUIDs are common identifiers and are not JSON-native.

A string representation is compact and reversible.

In [30]:
@json_format.register(uuid.UUID)
def _(obj):
    return {
        "__type__": "uuid",
        "value": str(obj),
    }

In [31]:
mixed = {
    10,
    "10",
    Decimal("10.0"),
    uuid.UUID("aaaaaaaa-aaaa-aaaa-aaaa-aaaaaaaaaaaa"),
}

In [32]:
first = json.dumps(
    {"values": mixed},
    default=json_format,
    sort_keys=True,
    separators=(",", ":"),
)

second = json.dumps(
    {"values": mixed},
    default=json_format,
    sort_keys=True,
    separators=(",", ":"),
)

assert first == second
print(first)

{"values":{"__type__":"set","items":["10",10,{"__type__":"uuid","value":"aaaaaaaa-aaaa-aaaa-aaaa-aaaaaaaaaaaa"}]}}


The goal is not to give the set a semantic order. The goal is to make the serialized text reproducible.

# Problem 10 — Encode bytes with Base64

JSON strings contain text, not arbitrary binary bytes.

Base64 is a standard reversible text encoding for binary data.

It is important to remember that Base64 is **not encryption**.

In [33]:
blob = b"\x00\x01hello\xff"

base64.b64encode(blob)

b'AAFoZWxsb/8='

The Base64 encoder returns bytes, so convert those ASCII bytes to a normal string.

In [34]:
encoded_blob = base64.b64encode(blob).decode("ascii")
encoded_blob

'AAFoZWxsb/8='

In [35]:
@json_format.register(bytes)
def _(obj):
    return {
        "__type__": "bytes",
        "encoding": "base64",
        "value": base64.b64encode(obj).decode("ascii"),
    }

In [36]:
print(
    json.dumps(
        {"blob": blob},
        default=json_format,
        indent=2,
    )
)

{
  "blob": {
    "__type__": "bytes",
    "encoding": "base64",
    "value": "AAFoZWxsb/8="
  }
}


# Problem 11 — Build a stricter central decoder

We now have several tagged types.

Let us centralize their decoding and make unknown tags fail explicitly.

In [37]:
def json_object_hook(obj):
    tag = obj.get("__type__")

    if tag is None:
        return obj

    if tag == "datetime":
        return parse_utc_datetime(obj["value"])

    if tag == "decimal":
        return Decimal(obj["value"])

    if tag == "set":
        return set(obj["items"])

    if tag == "uuid":
        return uuid.UUID(obj["value"])

    if tag == "bytes":
        if obj.get("encoding") != "base64":
            raise ValueError("Unsupported bytes encoding.")
        return base64.b64decode(obj["value"], validate=True)

    raise ValueError(f"Unknown JSON type tag: {tag!r}")

Test several nested values at once.

In [38]:
nested = {
    "id": uuid.UUID("bbbbbbbb-bbbb-bbbb-bbbb-bbbbbbbbbbbb"),
    "created_at": datetime(2026, 8, 7, 12, 0, tzinfo=timezone.utc),
    "amounts": [
        Decimal("1.10"),
        Decimal("2.20"),
    ],
    "metadata": {
        "raw": b"abc",
        "flags": {"x", "y"},
    },
}

nested_text = json.dumps(nested, default=json_format, indent=2)
print(nested_text)

{
  "id": {
    "__type__": "uuid",
    "value": "bbbbbbbb-bbbb-bbbb-bbbb-bbbbbbbbbbbb"
  },
  "created_at": {
    "__type__": "datetime",
    "value": "2026-08-07T12:00:00Z"
  },
  "amounts": [
    {
      "__type__": "decimal",
      "value": "1.10"
    },
    {
      "__type__": "decimal",
      "value": "2.20"
    }
  ],
  "metadata": {
    "raw": {
      "__type__": "bytes",
      "encoding": "base64",
      "value": "YWJj"
    },
    "flags": {
      "__type__": "set",
      "items": [
        "x",
        "y"
      ]
    }
  }
}


In [39]:
nested_copy = json.loads(nested_text, object_hook=json_object_hook)

print(type(nested_copy["id"]))
print(type(nested_copy["created_at"]))
print(type(nested_copy["amounts"][0]))
print(type(nested_copy["metadata"]["raw"]))
print(type(nested_copy["metadata"]["flags"]))

<class 'uuid.UUID'>
<class 'datetime.datetime'>
<class 'decimal.Decimal'>
<class 'bytes'>
<class 'set'>


This works recursively because inner tagged dictionaries are decoded before their containing dictionaries.

# Problem 12 — Let custom objects contain other custom objects

We will create a `Money` class containing a `Decimal`.

A useful design is to let the Money serializer return the Decimal directly.

The JSON encoder can then recursively call our formatter for that nested Decimal.

In [40]:
@dataclass(frozen=True)
class Money:
    amount: Decimal
    currency: str

In [41]:
@json_format.register(Money)
def _(obj):
    return {
        "__type__": "money",
        "amount": obj.amount,
        "currency": obj.currency,
    }

In [42]:
price = Money(Decimal("19.99"), "EUR")

money_text = json.dumps(price, default=json_format, indent=2)
print(money_text)

{
  "__type__": "money",
  "amount": {
    "__type__": "decimal",
    "value": "19.99"
  },
  "currency": "EUR"
}


Notice that the nested `amount` became its own tagged Decimal without the Money formatter manually converting it.

In [43]:
_previous_hook = json_object_hook

def json_object_hook(obj):
    if obj.get("__type__") == "money":
        return Money(
            amount=obj["amount"],
            currency=obj["currency"],
        )

    return _previous_hook(obj)

In [44]:
price_copy = json.loads(money_text, object_hook=json_object_hook)

assert price_copy == price
assert isinstance(price_copy.amount, Decimal)

print(price_copy)

Money(amount=Decimal('19.99'), currency='EUR')


# Problem 13 — Generic dataclass support with a whitelist

Writing a custom formatter for every dataclass can become repetitive.

We can generalize dataclass serialization, but decoding arbitrary class names from JSON would be unsafe.

So we will use an explicit registry of allowed dataclass types.

In [45]:
DATACLASS_REGISTRY = {}


def register_dataclass(cls):
    if not is_dataclass(cls):
        raise TypeError("Only dataclass types can be registered.")

    DATACLASS_REGISTRY[cls.__name__] = cls
    return cls

In [46]:
@register_dataclass
@dataclass
class User:
    name: str
    user_id: uuid.UUID


@register_dataclass
@dataclass
class AuditRecord:
    actor: User
    action: str
    created_at: datetime

Now use the generic `object` implementation of `singledispatch` as a controlled fallback.

It accepts only registered dataclass instances and rejects everything else.

In [47]:
@json_format.register(object)
def _(obj):
    if is_dataclass(obj) and not isinstance(obj, type):
        cls = type(obj)

        if cls.__name__ not in DATACLASS_REGISTRY:
            raise TypeError(f"Dataclass is not registered: {cls.__name__}")

        payload = {
            field.name: getattr(obj, field.name)
            for field in fields(obj)
        }

        return {
            "__type__": "dataclass",
            "class": cls.__name__,
            "fields": payload,
        }

    raise TypeError(f"Unsupported JSON type: {type(obj).__name__}")

In [48]:
record = AuditRecord(
    actor=User(
        name="Ada",
        user_id=uuid.UUID("cccccccc-cccc-cccc-cccc-cccccccccccc"),
    ),
    action="login",
    created_at=datetime(2026, 8, 7, 12, 30, tzinfo=timezone.utc),
)

record_text = json.dumps(record, default=json_format, indent=2)
print(record_text)

{
  "__type__": "dataclass",
  "class": "AuditRecord",
  "fields": {
    "actor": {
      "__type__": "dataclass",
      "class": "User",
      "fields": {
        "name": "Ada",
        "user_id": {
          "__type__": "uuid",
          "value": "cccccccc-cccc-cccc-cccc-cccccccccccc"
        }
      }
    },
    "action": "login",
    "created_at": {
      "__type__": "datetime",
      "value": "2026-08-07T12:30:00Z"
    }
  }
}


# Problem 14 — Reconstruct nested dataclasses safely

The dataclass decoder will:

1. read the class name,
2. look it up in our explicit registry,
3. reject unknown names,
4. construct the registered class.

We do not dynamically import or evaluate names from JSON.

In [49]:
_previous_hook_2 = json_object_hook

def json_object_hook(obj):
    if obj.get("__type__") == "dataclass":
        class_name = obj["class"]
        cls = DATACLASS_REGISTRY.get(class_name)

        if cls is None:
            raise ValueError(f"Unknown dataclass: {class_name!r}")

        return cls(**obj["fields"])

    return _previous_hook_2(obj)

In [50]:
record_copy = json.loads(record_text, object_hook=json_object_hook)

print(record_copy)
print(type(record_copy))
print(type(record_copy.actor))
print(type(record_copy.created_at))

AuditRecord(actor=User(name='Ada', user_id=UUID('cccccccc-cccc-cccc-cccc-cccccccccccc')), action='login', created_at=datetime.datetime(2026, 8, 7, 12, 30, tzinfo=datetime.timezone.utc))
<class '__main__.AuditRecord'>
<class '__main__.User'>
<class 'datetime.datetime'>


In [51]:
assert record_copy == record
print("Nested dataclass round trip passed.")

Nested dataclass round trip passed.


# Problem 15 — Add Enum support through a registry

Enums are another common domain type.

We will serialize:

- the enum class name,
- and the enum value.

Decoding will again use a whitelist.

In [52]:
class Status(Enum):
    NEW = "new"
    ACTIVE = "active"
    DISABLED = "disabled"


ENUM_REGISTRY = {
    "Status": Status,
}

In [53]:
@json_format.register(Enum)
def _(obj):
    return {
        "__type__": "enum",
        "enum": type(obj).__name__,
        "value": obj.value,
    }

In [54]:
_previous_hook_3 = json_object_hook

def json_object_hook(obj):
    if obj.get("__type__") == "enum":
        enum_name = obj["enum"]
        enum_cls = ENUM_REGISTRY.get(enum_name)

        if enum_cls is None:
            raise ValueError(f"Unknown Enum: {enum_name!r}")

        return enum_cls(obj["value"])

    return _previous_hook_3(obj)

In [55]:
status_text = json.dumps(
    {"status": Status.ACTIVE},
    default=json_format,
    indent=2,
)

print(status_text)

{
  "status": {
    "__type__": "enum",
    "enum": "Status",
    "value": "active"
  }
}


In [56]:
status_copy = json.loads(status_text, object_hook=json_object_hook)

assert status_copy["status"] is Status.ACTIVE
print(status_copy)

{'status': <Status.ACTIVE: 'active'>}


# Problem 16 — Why a strict fallback is safer than `vars()` or `str()`

It is tempting to say:

- if an object has attributes, serialize `vars(obj)`,
- otherwise call `str(obj)`.

That can accidentally leak internal state.

In [57]:
class Session:
    def __init__(self, user, access_token):
        self.user = user
        self.access_token = access_token
        self.cache = {"internal": True}

In [58]:
session = Session("ada", "SECRET-TOKEN")

vars(session)

{'user': 'ada', 'access_token': 'SECRET-TOKEN', 'cache': {'internal': True}}

A generic `vars()` fallback would expose the access token and cache.

Our strict formatter rejects this object because we have not explicitly chosen a public representation.

In [59]:
try:
    json.dumps(session, default=json_format)
except TypeError as ex:
    print(type(ex).__name__, ex)

TypeError Unsupported JSON type: Session


If serialization is needed, define exactly what may leave the object.

In [60]:
@json_format.register(Session)
def _(obj):
    return {
        "__type__": "session_public",
        "user": obj.user,
    }

In [61]:
session_text = json.dumps(session, default=json_format, indent=2)

print(session_text)

assert "SECRET-TOKEN" not in session_text
assert "cache" not in session_text

{
  "__type__": "session_public",
  "user": "ada"
}


# Problem 17 — Package the policy in a `JSONEncoder` subclass

Instead of passing `default=json_format` everywhere, some projects prefer one named encoder class.

In [62]:
class AdvancedJSONEncoder(json.JSONEncoder):
    def default(self, obj):
        try:
            return json_format(obj)
        except TypeError:
            return super().default(obj)

In [63]:
sample = {
    "created_at": datetime(2026, 8, 7, 12, 0, tzinfo=timezone.utc),
    "amount": Decimal("100.25"),
    "status": Status.NEW,
}

print(
    json.dumps(
        sample,
        cls=AdvancedJSONEncoder,
        indent=2,
    )
)

{
  "created_at": {
    "__type__": "datetime",
    "value": "2026-08-07T12:00:00Z"
  },
  "amount": {
    "__type__": "decimal",
    "value": "100.25"
  },
  "status": {
    "__type__": "enum",
    "enum": "Status",
    "value": "new"
  }
}


The class does not replace our singledispatch logic. It simply wraps it in a reusable JSON encoder policy.

# Problem 18 — Create public `dumps` and `loads` façade functions

Application code should not need to remember every serialization option.

We will centralize the policy behind two small helpers.

In [64]:
def dumps(obj, *, pretty=False):
    options = {
        "cls": AdvancedJSONEncoder,
        "ensure_ascii": False,
        "sort_keys": True,
        "allow_nan": False,
    }

    if pretty:
        options["indent"] = 2
    else:
        options["separators"] = (",", ":")

    return json.dumps(obj, **options)


def loads(text):
    return json.loads(
        text,
        object_hook=json_object_hook,
        parse_float=Decimal,
    )

`allow_nan=False` is deliberate.

JSON does not standardize `NaN`, positive infinity, or negative infinity as normal number literals.

In [65]:
for value in [math.nan, math.inf, -math.inf]:
    try:
        dumps({"value": value})
    except ValueError as ex:
        print("Rejected:", value, "->", ex)

Rejected: nan -> Out of range float values are not JSON compliant: nan
Rejected: inf -> Out of range float values are not JSON compliant: inf
Rejected: -inf -> Out of range float values are not JSON compliant: -inf


# Problem 19 — Version a durable JSON document

JSON stored on disk or sent between long-lived services can outlive the code that originally created it.

Suppose version 1 stores one `name` field, while version 2 stores `first_name` and `last_name`.

Add an explicit schema version instead of guessing from field shape.

In [66]:
def make_envelope(payload, version):
    return {
        "schema": "user-profile",
        "version": version,
        "payload": payload,
    }


v1 = make_envelope(
    {"name": "Ada Lovelace"},
    version=1,
)

print(json.dumps(v1, indent=2))

{
  "schema": "user-profile",
  "version": 1,
  "payload": {
    "name": "Ada Lovelace"
  }
}


Now write a migration to version 2.

In [67]:
CURRENT_SCHEMA_VERSION = 2


def migrate(document):
    if document.get("schema") != "user-profile":
        raise ValueError("Unexpected schema.")

    version = document.get("version")

    if version == 1:
        document = dict(document)
        payload = dict(document["payload"])

        first, _, last = payload.pop("name").partition(" ")

        payload["first_name"] = first
        payload["last_name"] = last

        document["payload"] = payload
        document["version"] = 2
        version = 2

    if version != CURRENT_SCHEMA_VERSION:
        raise ValueError(f"Unsupported schema version: {version!r}")

    return document

In [68]:
v2 = migrate(v1)

print(json.dumps(v2, indent=2))

{
  "schema": "user-profile",
  "version": 2,
  "payload": {
    "first_name": "Ada",
    "last_name": "Lovelace"
  }
}


Explicit versioning makes evolution testable and keeps migrations intentional.

# Problem 20 — Use JSON Lines for an event stream

A giant JSON array is awkward for append-only logs.

JSON Lines stores one complete JSON value per line.

In [69]:
events = [
    {
        "event": "login",
        "time": datetime(2026, 8, 7, 12, 0, tzinfo=timezone.utc),
    },
    {
        "event": "purchase",
        "amount": Decimal("9.99"),
        "time": datetime(2026, 8, 7, 12, 1, tzinfo=timezone.utc),
    },
]

In [70]:
def write_jsonl(records, file_obj):
    for record in records:
        file_obj.write(dumps(record))
        file_obj.write("\n")


def read_jsonl(file_obj):
    for line_number, line in enumerate(file_obj, start=1):
        line = line.strip()

        if not line:
            continue

        try:
            yield loads(line)
        except Exception as ex:
            raise ValueError(
                f"Invalid JSONL record at line {line_number}: {ex}"
            ) from ex

In [71]:
buffer = StringIO()

write_jsonl(events, buffer)

print(buffer.getvalue())

{"event":"login","time":{"__type__":"datetime","value":"2026-08-07T12:00:00Z"}}
{"amount":{"__type__":"decimal","value":"9.99"},"event":"purchase","time":{"__type__":"datetime","value":"2026-08-07T12:01:00Z"}}



In [72]:
buffer.seek(0)
events_copy = list(read_jsonl(buffer))

assert events_copy == events
print(events_copy)

[{'event': 'login', 'time': datetime.datetime(2026, 8, 7, 12, 0, tzinfo=datetime.timezone.utc)}, {'amount': Decimal('9.99'), 'event': 'purchase', 'time': datetime.datetime(2026, 8, 7, 12, 1, tzinfo=datetime.timezone.utc)}]


# Problem 21 — Reject unknown type tags

If a typed decoder sees a tag it does not understand, silently returning the dictionary can hide a protocol mismatch.

Our decoder is intentionally strict.

In [73]:
unknown = '''
{
  "__type__": "mystery-object",
  "value": 123
}
'''

In [74]:
try:
    loads(unknown)
except ValueError as ex:
    print(type(ex).__name__, ex)

ValueError Unknown JSON type tag: 'mystery-object'


Failing early makes corrupted data and incompatible schema versions easier to detect.

# Problem 22 — Never reconstruct arbitrary class names from input

A dangerous design would evaluate or dynamically import the class name found in JSON.

That turns data parsing into code execution.

Our dataclass registry avoids that.

In [75]:
malicious_or_unknown = '''
{
  "__type__": "dataclass",
  "class": "os.system",
  "fields": {
    "command": "echo should-not-run"
  }
}
'''

In [76]:
try:
    loads(malicious_or_unknown)
except ValueError as ex:
    print("Safely rejected:", ex)

Safely rejected: Unknown dataclass: 'os.system'


Only classes explicitly added to `DATACLASS_REGISTRY` can be constructed.

# Problem 23 — Type-tag collisions are part of protocol design

Our decoder reserves the key `__type__`.

But ordinary user data might legitimately contain:

```python
{"__type__": "decimal", "value": "999"}
```

The decoder would mistake it for a typed value.

In [77]:
ordinary_user_data = {
    "__type__": "decimal",
    "value": "999",
}

ordinary_user_data

{'__type__': 'decimal', 'value': '999'}

One improvement is to use a dedicated wrapper with a more recognizable protocol boundary.

In [78]:
def wrap_tag(type_name, payload):
    return {
        "__python_json__": {
            "type": type_name,
            "payload": payload,
        }
    }

In [79]:
wrap_tag(
    "decimal",
    {"value": "999"},
)

{'__python_json__': {'type': 'decimal', 'payload': {'value': '999'}}}

This does not mathematically eliminate collisions, but it reduces accidental ones.

For high-assurance systems, validate the entire document against an explicit schema.

# Problem 24 — Why tuples need preprocessing

Tuples reveal an important limitation of `default=`.

Python's built-in JSON encoder already knows how to serialize a tuple: it converts it to a JSON array.

So the tuple never reaches our `default` callback.

In [80]:
original_tuple = (1, 2, 3)

tuple_text = json.dumps(original_tuple)
tuple_copy = json.loads(tuple_text)

print(tuple_text)
print(tuple_copy)
print(type(tuple_copy))

[1, 2, 3]
[1, 2, 3]
<class 'list'>


If tuple-vs-list distinction matters, we need to transform the object graph before encoding.

In [81]:
def preprocess_tuples(obj):
    if isinstance(obj, tuple):
        return {
            "__type__": "tuple",
            "items": [
                preprocess_tuples(item)
                for item in obj
            ],
        }

    if isinstance(obj, list):
        return [
            preprocess_tuples(item)
            for item in obj
        ]

    if isinstance(obj, dict):
        return {
            key: preprocess_tuples(value)
            for key, value in obj.items()
        }

    return obj

In [82]:
mixed_sequence = {
    "list": [1, 2],
    "tuple": (1, 2),
}

preprocess_tuples(mixed_sequence)

{'list': [1, 2], 'tuple': {'__type__': 'tuple', 'items': [1, 2]}}

The lesson is broader than tuples:

> `default=` only handles objects the standard encoder does not already know how to transform.

# Problem 25 — Circular references cannot be represented directly

JSON naturally represents trees.

Python containers can form graphs containing cycles.

In [83]:
cyclic = []
cyclic.append(cyclic)

In [84]:
try:
    json.dumps(cyclic)
except ValueError as ex:
    print(type(ex).__name__, ex)

ValueError Circular reference detected


If shared identity or cycles must be preserved, the JSON format needs an explicit reference model, for example:

```python
{"$ref": "object-17"}
```

or a normalized table of objects keyed by IDs.

For normal API payloads, rejecting cycles is usually simpler.

# Problem 26 — Capstone nested domain object

We will now combine most of the pieces in one realistic object.

The serializer will need to recurse through:

- a registered dataclass,
- another nested dataclass,
- UUIDs,
- Decimal,
- Enum,
- datetime,
- bytes,
- and a set.

In [85]:
@register_dataclass
@dataclass
class PaymentEvent:
    event_id: uuid.UUID
    actor: User
    amount: Decimal
    status: Status
    created_at: datetime
    raw_receipt: bytes
    labels: set

In [86]:
event = PaymentEvent(
    event_id=uuid.UUID("dddddddd-dddd-dddd-dddd-dddddddddddd"),
    actor=User(
        name="Grace",
        user_id=uuid.UUID("eeeeeeee-eeee-eeee-eeee-eeeeeeeeeeee"),
    ),
    amount=Decimal("1000000.00000001"),
    status=Status.ACTIVE,
    created_at=datetime(2026, 8, 7, 14, 30, tzinfo=timezone.utc),
    raw_receipt=b"receipt:\x00\x01",
    labels={"finance", "reviewed"},
)

Before serializing, trace the recursive behavior mentally:

1. `PaymentEvent` becomes a tagged dataclass dictionary.
2. The nested `User` becomes another tagged dataclass.
3. UUIDs become tagged UUID dictionaries.
4. Decimal becomes a tagged decimal.
5. Enum becomes a tagged enum.
6. Datetime becomes a UTC tagged timestamp.
7. Bytes become Base64.
8. Set becomes a deterministic tagged list.

In [87]:
event_text = dumps(event, pretty=True)
print(event_text)

{
  "__type__": "dataclass",
  "class": "PaymentEvent",
  "fields": {
    "actor": {
      "__type__": "dataclass",
      "class": "User",
      "fields": {
        "name": "Grace",
        "user_id": {
          "__type__": "uuid",
          "value": "eeeeeeee-eeee-eeee-eeee-eeeeeeeeeeee"
        }
      }
    },
    "amount": {
      "__type__": "decimal",
      "value": "1000000.00000001"
    },
    "created_at": {
      "__type__": "datetime",
      "value": "2026-08-07T14:30:00Z"
    },
    "event_id": {
      "__type__": "uuid",
      "value": "dddddddd-dddd-dddd-dddd-dddddddddddd"
    },
    "labels": {
      "__type__": "set",
      "items": [
        "finance",
        "reviewed"
      ]
    },
    "raw_receipt": {
      "__type__": "bytes",
      "encoding": "base64",
      "value": "cmVjZWlwdDoAAQ=="
    },
    "status": {
      "__type__": "enum",
      "enum": "Status",
      "value": "active"
    }
  }
}


In [88]:
event_copy = loads(event_text)

assert event_copy == event
assert isinstance(event_copy, PaymentEvent)
assert isinstance(event_copy.actor, User)
assert isinstance(event_copy.event_id, uuid.UUID)
assert isinstance(event_copy.amount, Decimal)
assert isinstance(event_copy.status, Status)
assert isinstance(event_copy.created_at, datetime)
assert isinstance(event_copy.raw_receipt, bytes)
assert isinstance(event_copy.labels, set)

print("Full capstone round trip passed.")

Full capstone round trip passed.


# Problem 27 — Test failure modes explicitly

Successful examples are not enough.

A production serializer should also prove that bad inputs fail in controlled ways.

### Case 1 — unsupported object

In [89]:
class Unsupported:
    pass


try:
    dumps(Unsupported())
except TypeError as ex:
    print("PASS:", ex)

PASS: Object of type Unsupported is not JSON serializable


### Case 2 — naive datetime

In [90]:
try:
    dumps(datetime(2026, 8, 7, 12, 0))
except ValueError as ex:
    print("PASS:", ex)

PASS: Naive datetime is not allowed.


### Case 3 — malformed Base64

In [91]:
bad_base64 = '''
{
  "__type__": "bytes",
  "encoding": "base64",
  "value": "***"
}
'''

try:
    loads(bad_base64)
except ValueError as ex:
    print("PASS:", ex)

PASS: Only base64 data is allowed


### Case 4 — unknown Enum

In [92]:
bad_enum = '''
{
  "__type__": "enum",
  "enum": "UnknownEnum",
  "value": "x"
}
'''

try:
    loads(bad_enum)
except ValueError as ex:
    print("PASS:", ex)

PASS: Unknown Enum: 'UnknownEnum'


Negative tests are especially important when the serializer is responsible for security boundaries, persistence, or communication between independently deployed systems.

# Problem 28 — Round-trip invariants

A useful serializer property is:

```python
loads(dumps(value)) == value
```

For typed serialization we often also care that the exact Python type survives.

In [93]:
round_trip_values = [
    Decimal("0"),
    Decimal("-999999999999999.0001"),
    uuid.UUID(int=0),
    b"",
    bytes(range(8)),
    set(),
    {"a", "b", "c"},
    datetime(2026, 1, 1, tzinfo=timezone.utc),
    Status.DISABLED,
    User(
        name="Lin",
        user_id=uuid.UUID("ffffffff-ffff-ffff-ffff-ffffffffffff"),
    ),
]

In [94]:
for index, value in enumerate(round_trip_values, start=1):
    restored = loads(dumps(value))

    assert restored == value, (index, value, restored)
    assert type(restored) is type(value), (
        index,
        type(value).__name__,
        type(restored).__name__,
    )

print(f"{len(round_trip_values)} round-trip invariants passed.")

10 round-trip invariants passed.


Example-based tests show that particular inputs work.

Invariant-driven tests help express what the serializer promises across a whole category of values.

# Final Review

The main lesson is that custom JSON serialization is not simply about converting objects to text.

It is about designing a **reversible, explicit, stable, and safe representation**.

A strong design usually makes deliberate decisions about:

- which Python types are allowed,
- which are rejected,
- how timezone information is represented,
- how numeric precision is preserved,
- whether ordering must be deterministic,
- how custom classes are identified,
- how classes are safely reconstructed,
- what schema version is being used,
- and what malformed inputs should do.

## Recommended production habits

1. Prefer JSON-native structures at public service boundaries whenever possible.
2. Use explicit tagged values only when typed round trips are genuinely needed.
3. Reject unsupported objects instead of silently calling `str`.
4. Use aware UTC timestamps.
5. Preserve `Decimal` values intentionally.
6. Whitelist reconstructable classes and Enums.
7. Never use `eval()` or arbitrary dynamic imports on type names from JSON.
8. Make ordering deterministic when serialized text is hashed, signed, or snapshot-tested.
9. Version durable formats.
10. Test both successful round trips and expected failures.

## Further exercises

Extend the codec with:

- `Fraction`
- `Path`
- `frozenset`
- `date`
- `time`
- a safer wrapper-based tagging protocol
- schema validation
- atomic file writes
- canonical JSON for hashing
- property-based tests using Hypothesis
- an explicit `$ref` system for graphs